# 04 · Calibração e treino da matriz (PPO em lote)

Roda jobs de **calibração** (β₁ por altitude, κ e α, só no treino da dobra) e de **treino** (PPO, PPO-CA, PPO-CA-ALT) no ambiente realista em lote na GPU. O tipo vem da chave `kind` do YAML. O treino é retomável (checkpoint a cada ≤ 15 min), registra passos/s em `progress.csv` (**gate G4: ≥ 5.000 passos/s**) e aplica o gate de aprendizado aos 25%: se falhar, a pasta ganha o sufixo `_INVALID_GATE` e o `DONE.json` sai com `status: INVALID_GATE`.

Antes: rode `00_setup.ipynb`. O job é **retomável**: se a sessão cair, rode o setup e este notebook de novo com o mesmo `JOB_ID`.

In [ ]:
JOB_ID = 'calibrate_A_dinov2'
import pandas as pd
jobs = pd.read_csv('/content/repo/colab/jobs_v3.csv')
job = jobs.set_index('job_id').loc[JOB_ID]
JOB_CONFIG = job.config_path
print(job)

In [ ]:
import subprocess, sys
cmd = [sys.executable, '-m', 'dal.v3.jobs', '--job-config', JOB_CONFIG, '--job-id', JOB_ID,
       '--out-root', '/content/drive/MyDrive/drone_v3', '--colab-out', 'colab_out']
print(subprocess.run(cmd, cwd='/content/repo', check=True).returncode)

In [ ]:
import json
done = json.load(open(f'/content/repo/colab_out/{JOB_ID}/DONE.json'))
print(json.dumps(done, indent=1))
main_number = str(done['metrics'].get('crops_per_second', ''))[:12]
subprocess.run([sys.executable, 'scripts/colab_publish_v3.py', '--job-id', JOB_ID,
                '--main-number', f'crops/s {main_number}'], cwd='/content/repo', check=True)